# RG3 RH 가스: 변동성 증가·반복 및 동시 이벤트
가스17개 전체 유지. 학습 정상에서만 이벤트 기준을 결정하고 같은 RF/분할로 평가한다.

In [1]:
from pathlib import Path
import json, subprocess
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
exec(''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source']).split('for feature in features:')[0])
allshots=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=allshots.loc[allshots['product'].eq('RG3')&(allshots.PassOrFail.eq(0)|allshots.Reason.str.contains('가스',na=False))].sort_values('TimeStamp').reset_index(drop=True)
assert len(d)==583 and d.PassOrFail.sum()==17
rh=labels.loc[labels['product'].eq('RG3')&labels.side.eq('RH')]
check=d[keys+['PassOrFail']].merge(rh[keys+['PassOrFail']],on=keys,validate='one_to_one')
assert check.PassOrFail_x.equals(check.PassOrFail_y)
sensors=['Max_Injection_Pressure','Average_Back_Pressure','Cushion_Position']
floors=np.array([.1,.1,.01]) # recording resolution, not optimized on labels
segments={k:g.sort_values('TimeStamp').reset_index(drop=True) for k,g in observed.loc[observed['product'].eq('RG3')].groupby('history_segment')}
recent=np.full((len(d),10,3),np.nan)
older=np.full((len(d),20,3),np.nan)
current_delta=np.full((len(d),3),np.nan)
for n,r in d.iterrows():
    g=segments[r.history_segment]
    i=int(np.searchsorted(g.TimeStamp.values,np.datetime64(r.TimeStamp),side='left'))
    # Current is excluded from both history windows. 31 prior observations give 30 increments.
    p=g.iloc[max(0,i-31):i][sensors].to_numpy(float)
    dif=np.diff(p,axis=0)
    if len(dif)>=10:recent[n]=dif[-10:]
    if len(dif)>=30:older[n]=dif[-30:-10]
    if i>=1:current_delta[n]=r[sensors].to_numpy(float)-g.iloc[i-1][sensors].to_numpy(float)
y=d.PassOrFail.to_numpy(int)
blocks=d.comparison_block.drop_duplicates().tolist()
baseline=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
models={
'baseline_std4':baseline,
'new_temporal3':['pressure_variability_growth','pressure_large_change_count','multi_sensor_change_count'],
'pressure_level_plus_temporal4':['Max_Injection_Pressure','Average_Back_Pressure','pressure_variability_growth','multi_sensor_change_count'],
'raw_pressure2_control':['Max_Injection_Pressure','Average_Back_Pressure']}
def make_fold_features(tr):
    # Event thresholds derived from training labeled normals ONLY.
    normal=tr & (y==0)
    thresholds=[]
    for j in range(3):
        z=np.abs(current_delta[normal,j]);z=z[np.isfinite(z)]
        thresholds.append(max(float(np.quantile(z,.95)),floors[j]) if len(z) else floors[j])
    thresholds=np.array(thresholds)
    x=d.copy()
    valid_recent=np.isfinite(recent).all(axis=(1,2))
    valid_long=np.isfinite(older).all(axis=(1,2))&valid_recent
    growth=np.full(len(d),np.nan)
    growth[valid_long]=np.minimum(np.std(recent[valid_long,:,0],axis=1)/np.maximum(np.std(older[valid_long,:,0],axis=1),floors[0]),20)
    event=np.abs(recent)>thresholds[None,None,:]+1e-8
    x['pressure_variability_growth']=growth
    x['pressure_large_change_count']=np.where(valid_recent,event[:,:,0].sum(axis=1),np.nan)
    x['multi_sensor_change_count']=np.where(valid_recent,(event.sum(axis=2)>=2).sum(axis=1),np.nan)
    return x,thresholds
coverage=pd.DataFrame({'group':np.where(y==1,'gas','normal'),'past10_complete':np.isfinite(recent).all(axis=(1,2)),'past30_complete':np.isfinite(older).all(axis=(1,2)) & np.isfinite(recent).all(axis=(1,2))}).groupby('group').agg(shots=('past10_complete','size'),past10_complete=('past10_complete','sum'),past30_complete=('past30_complete','sum'))
coverage.to_csv(tables/'rg3_temporal_event_coverage.csv',encoding='utf-8-sig')
metrics=[]; predictions=[]; thresholds_log=[]; features_log=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    probs={name:np.full(len(d),np.nan) for name in models}
    for j,b in enumerate(blocks):
        te=d.comparison_block.eq(b).to_numpy()
        tr=~te if scheme=='leave_one_block_out' else d.comparison_block.isin(blocks[:j]).to_numpy()
        if len(np.unique(y[tr]))<2:continue
        x,thresholds=make_fold_features(tr)
        for s,t in zip(sensors,thresholds):
            thresholds_log.append(dict(scheme=scheme,held_out_block=b,sensor=s,absolute_delta_threshold=t,train_normals=int((tr&(y==0)).sum())))
        for i in np.flatnonzero(te):
            features_log.append(dict(scheme=scheme,TimeStamp=d.TimeStamp.iloc[i],comparison_block=b,actual=int(y[i]),**{f:x[f].iloc[i] for f in models['new_temporal3']}))
        for name,fs in models.items():
            m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1))
            m.fit(x.loc[tr,fs],y[tr]);probs[name][te]=m.predict_proba(x.loc[te,fs])[:,1]
    for name,p in probs.items():
        ok=np.isfinite(p);pred=p[ok]>=.5
        tn,fp,fn,tp=confusion_matrix(y[ok],pred,labels=[0,1]).ravel()
        selected=np.zeros(len(d),bool)
        for b in blocks:
            ids=np.flatnonzero(ok & d.comparison_block.eq(b).to_numpy())
            if len(ids):
                tie=np.random.RandomState(42).rand(len(ids))
                selected[ids[np.lexsort((tie,-p[ids]))[:int(np.ceil(len(ids)*.1))]]]=True
        r=dict(scheme=scheme,model=name,inputs=len(models[name]),shots=int(ok.sum()),gas=int(y[ok].sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(y[ok],p[ok]),F1=f1_score(y[ok],pred,zero_division=0),top10_selected=int(selected.sum()),top10_TP=int(y[selected].sum()),top10_FP=int((y[selected]==0).sum()))
        metrics.append(r);print(r,flush=True)
        for i in np.flatnonzero(ok):
            predictions.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],comparison_block=d.comparison_block.iloc[i],actual=int(y[i]),probability=p[i],prediction=int(p[i]>=.5),selected_top10=bool(selected[i])))
mt=pd.DataFrame(metrics)
mt.to_csv(tables/'rg3_temporal_event_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(predictions).to_csv(tables/'rg3_temporal_event_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(thresholds_log).to_csv(tables/'rg3_temporal_event_training_thresholds.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(features_log).to_csv(tables/'rg3_temporal_event_fold_features.csv',index=False,encoding='utf-8-sig')
assert np.isclose(mt.loc[(mt.scheme=='leave_one_block_out')&(mt.model=='baseline_std4'),'AP'].iloc[0],.13871113959502832)
def tab(frame):
    cols=['model','inputs','shots','gas','TP','FP','FN','TN','AP','top10_TP','top10_FP','top10_selected']
    out=['| '+' | '.join(cols)+' |','|'+'|'.join(['---']*len(cols))+'|']
    for _,r in frame.iterrows():out.append('| '+' | '.join(f'{r[c]:.4f}' if c=='AP' else str(r[c]) for c in cols)+' |')
    return out
lines=['# RG3 가스: 변동성 증가·반복 이벤트·동시 변화','',
'## 이번에 새로 확인한 정보','',
'- pressure_variability_growth: 직전10개 압력 변화량 표준편차 / 그 앞20개 변화량 표준편차. 분모 하한0.1, 비율 상한20. 값 수준이 아닌 변화량의 불안정 증가.',
'- pressure_large_change_count: 직전10개 변화 중 압력 변화가 학습 정상 변화량 절댓값95% 분위수보다 큰 횟수.',
'- multi_sensor_change_count: 직전10개 변화에서 사출압력·평균배압·쿠션 중2개 이상이 동시에 각 큰변화 기준을 넘은 횟수.',
'- 95% 기준은 테스트 구간을 제외한 학습 정상에서만 계산. 기준 하한은 압력·배압0.1, 쿠션0.01(기록 단위). 이벤트는 현재Shot을 제외한 과거에서 센다.',
'- 기존 단일 delta1/과거 표준편차/평균편차/경로 효율/센서 상관을 새 이름으로 재사용한 것과 달리 변화량 분산의 상대 증가와 반복·동시 이벤트를 세는 후보이다.',
'','## 평가','',
'- RG3 RH 정상566 + 가스17, 총583개 유지. 미성형8개 제외, LH 합치지 않음.',
'- 기존4변수·새3변수·현재압력2개+새패턴2개를 비교. 현재압력2개만의 대조군도 함께 넣어 원값 추가 효과를 구별.',
'- 같은 RF200, depth8, leaf3, balanced_subsample, seed42. 확률0.5 고정, 결측은 학습 중앙값으로 대체.',
'- 구간 전체 제외5회: 기존과 직접 비교. 과거구간만 학습: 후반440개/가스13개 별도 평가.',
'- 30분 이력 초기화 규칙은 실제Run으로 확정하지 않는다. 충분한 이력이 없는 행도 제거하지 않음.',
'- 상위10%는 각 평가구간별 고정 검사 예산. 미래의 구간 전체 순위를 사용하므로 배치 검사 순위이며 실시간 경보 성능과 다름.',
'- 반복 탐색에서 같은 데이터를 재사용하고 있으므로 독립 최종 성능을 주장하지 않음. Timestamp/연속 관측이 테스트에 제공되는지도 추가 확인 필요.',
'','## 이력 확보','',coverage.to_string(),'']
for scheme,g in mt.groupby('scheme',sort=False):lines+=['## '+scheme,'']+tab(g)+['']
for scheme,g in mt.groupby('scheme',sort=False):
    base=g.loc[g.model.eq('baseline_std4')].iloc[0]
    for _,r in g.loc[g.model.isin(['new_temporal3','pressure_level_plus_temporal4'])].iterrows():
        lines.append(f'- {scheme} / {r.model}: 기존대비 AP {r.AP-base.AP:+.4f}, TP {r.TP-base.TP:+d}, FP {r.FP-base.FP:+d}, 같은 검사예산 발견수 {r.top10_TP-base.top10_TP:+d}.')
lines+=['','## 해석','',
'확률0.5에서 TP를 늘려도 FP가 크게 늘면 실용적 개선이라고 하지 않는다. 검사 순위와 자동판정은 별개이며, 설정을 전체평가 결과로 선택한 뒤 최고값만 보고하면 낙관적이다. 추가 임계값 조정은 이번 실험에서 하지 않았다. 비라벨은 센서 이력일 뿐 정상 정답이나 가짜 불량 라벨로 사용하지 않았다.',
'','## 산출물','',
'- notebooks/21_rg3_temporal_event_features.ipynb',
'- reports/tables/rg3_temporal_event_metrics.csv',
'- reports/tables/rg3_temporal_event_predictions.csv',
'- reports/tables/rg3_temporal_event_training_thresholds.csv',
'- reports/tables/rg3_temporal_event_fold_features.csv']
(root/'reports/rg3_temporal_event_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(coverage.to_string())


             scheme                         model  inputs  shots  gas  TP  FP  FN  TN       AP       F1  top10_selected  top10_TP  top10_FP
leave_one_block_out                 baseline_std4       4    583   17   1   6  16 560 0.138711 0.083333              62         6        56
leave_one_block_out                 new_temporal3       3    583   17   0  18  17 548 0.032486 0.000000              62         1        61
leave_one_block_out pressure_level_plus_temporal4       4    583   17   0   3  17 563 0.032139 0.000000              62         2        60
leave_one_block_out         raw_pressure2_control       2    583   17   2  71  15 495 0.036389 0.044444              62         1        61
   past_blocks_only                 baseline_std4       4    440   13   0   2  13 425 0.113302 0.000000              46         4        42
   past_blocks_only                 new_temporal3       3    440   13   0  15  13 412 0.025143 0.000000              46         0        46
   past_blocks_only 